## Projeto Prático 2: Detecção de Objetos com YOLO e Transferência de Aprendizado

A seção anterior demonstrou, por meio da rede *Faster R-CNN*, um modelo de detecção pré-treinado aplicado diretamente, sem qualquer etapa de re-treinamento. O *Faster R-CNN* pertence à família dos detectores **de dois estágios**: uma primeira rede propõe regiões candidatas à presença de um objeto — as *region proposals* — e uma segunda rede classifica e refina cada uma dessas regiões (Ren et al., 2015).

A família **YOLO** (*You Only Look Once*) segue uma filosofia distinta, denominada **de estágio único**: a imagem é dividida conceitualmente em uma grade, e uma única rede convolucional estima, em uma única passagem direta (*forward pass*), as caixas delimitadoras (*bounding boxes*), o grau de confiança de presença de um objeto e a classe correspondente a cada região da grade (Redmon et al., 2016). A unificação das etapas de proposição e de classificação em um único cálculo é o fator que possibilita o processamento em tempo real (*real-time*), característica que viabiliza aplicações como monitoramento por vídeo e robótica móvel.

Nesta seção, o princípio de **transferência de aprendizado** (*transfer learning*) — a reutilização de representações aprendidas em uma tarefa de origem para acelerar o aprendizado de uma tarefa de destino distinta (Pan & Yang, 2010) — é aplicado a uma tarefa de detecção. Parte-se de um modelo YOLO pré-treinado no conjunto de dados COCO, composto por 80 categorias de objetos naturais (Lin et al., 2014), e realiza-se o ajuste fino (*fine-tuning*) para a localização e a classificação de objetos geometricamente definidos — triângulos, quadrados, estrelas, entre outros —, variando em cor, dimensão, ângulo de rotação e sob degradação por ruído do tipo sal e pimenta (*salt-and-pepper*).

### Bloco 1: Definição das Formas, Injeção de Ruído e Exibição das Amostras do *Dataset*

O cenário reproduz um problema característico da visão computacional industrial ou robótica: a partir de imagens contendo formas geométricas capturadas sob degradação de sensor, prepara-se um detector para delimitar e classificar cada objeto em nove categorias, listadas na variável `CLASSES`.

Duas etapas de geração de dados antecedem o treinamento. Primeiro, funções geométricas específicas constroem as coordenadas de cada forma: `poligono_regular` gera polígonos regulares (triângulo, quadrado, pentágono, hexágono e heptágono) a partir do número de lados e do ângulo de rotação; `poligono_estrela` gera a estrela de cinco pontas alternando raios interno e externo; e `poligono_cruz` constrói a cruz por meio de uma matriz de rotação. A função `desenha_objeto` invoca a rotina correspondente a cada classe, desenha a forma sobre a imagem com a biblioteca `cv2` e retorna os limites da caixa delimitadora (*bounding box*) — o menor retângulo que envolve completamente o objeto.

Segundo, a função `adiciona_ruido_sal_pimenta` introduz uma degradação denominada ruído sal e pimenta (*salt-and-pepper noise*): um subconjunto aleatório de pixels da imagem é substituído por valores extremos, brancos (255) ou pretos (0), simulando falhas impulsivas comuns em sensores óticos e em canais de transmissão de imagem (Gonzalez & Woods, 2018). Na função `gera_imagem_ruidosa`, esse ruído é aplicado com intensidade de 5% dos pixels da imagem.

A @fig-09-yolo-amostras-iniciais apresenta cinco amostras do conjunto sintético, já degradadas pelo ruído, com as caixas delimitadoras sobrepostas no formato de anotação YOLO — quíntupla $(c_i, x_c, y_c, w, h)$, em que $c_i$ identifica a classe e as demais coordenadas, normalizadas entre 0 e 1, descrevem o centro e as dimensões da caixa —, exibidas por meio das funções `mm.showBoundBox()` e `mm.show()`, pertencentes à biblioteca didática `morph.py`.

In [ ]:
import numpy as np
import cv2
import random
import morph as mm  # biblioteca didática de visão computacional

CLASSES = [
    'Triangle', 'Square', 'Pentagon', 'Hexagon',
    'Heptagon', 'Circle', 'Ellipse', 'Star', 'Cross'
]
N_LADOS = {'Triangle': 3, 'Square': 4, 'Pentagon': 5, 'Hexagon': 6, 'Heptagon': 7}


def poligono_regular(cx, cy, r, n_lados, rot_graus):
    ang0 = np.deg2rad(rot_graus - 90)
    angs = ang0 + 2 * np.pi * np.arange(n_lados) / n_lados
    return np.stack([cx + r * np.cos(angs), cy + r * np.sin(angs)], axis=1)


def poligono_estrela(cx, cy, r_externo, rot_graus, n_pontas=5):
    r_interno = r_externo * 0.45
    ang0 = np.deg2rad(rot_graus - 90)
    angs = ang0 + np.pi * np.arange(2 * n_pontas) / n_pontas
    raios = np.where(np.arange(2 * n_pontas) % 2 == 0, r_externo, r_interno)
    return np.stack([cx + raios * np.cos(angs), cy + raios * np.sin(angs)], axis=1)


def poligono_cruz(cx, cy, r, rot_graus, espessura_rel=0.35):
    w = r * espessura_rel
    base = np.array([
        (-w, -r), (w, -r), (w, -w), (r, -w), (r, w), (w, w),
        (w, r), (-w, r), (-w, w), (-r, w), (-r, -w), (-w, -w),
    ])
    theta = np.deg2rad(rot_graus)
    R = np.array([[np.cos(theta), -np.sin(theta)], [np.sin(theta), np.cos(theta)]])
    return base @ R.T + np.array([cx, cy])


def desenha_objeto(img, classe_idx, cx, cy, tamanho, rotacao, cor):
    nome = CLASSES[classe_idx]
    if nome in N_LADOS:
        pts = poligono_regular(cx, cy, tamanho, N_LADOS[nome], rotacao)
        cv2.fillPoly(img, [pts.astype(np.int32)], cor)
        xs, ys = pts[:, 0], pts[:, 1]
    elif nome == 'Star':
        pts = poligono_estrela(cx, cy, tamanho, rotacao)
        cv2.fillPoly(img, [pts.astype(np.int32)], cor)
        xs, ys = pts[:, 0], pts[:, 1]
    elif nome == 'Cross':
        pts = poligono_cruz(cx, cy, tamanho, rotacao)
        cv2.fillPoly(img, [pts.astype(np.int32)], cor)
        xs, ys = pts[:, 0], pts[:, 1]
    elif nome == 'Circle':
        cv2.circle(img, (int(cx), int(cy)), int(tamanho), cor, -1)
        xs, ys = np.array([cx - tamanho, cx + tamanho]), np.array([cy - tamanho, cy + tamanho])
    else:  # Ellipse
        eixo = (int(tamanho), int(tamanho * 0.6))
        cv2.ellipse(img, (int(cx), int(cy)), eixo, rotacao, 0, 360, cor, -1)
        ang = np.deg2rad(rotacao)
        dx = np.hypot(eixo[0] * np.cos(ang), eixo[1] * np.sin(ang))
        dy = np.hypot(eixo[0] * np.sin(ang), eixo[1] * np.cos(ang))
        xs, ys = np.array([cx - dx, cx + dx]), np.array([cy - dy, cy + dy])
    return xs.min(), ys.min(), xs.max(), ys.max()


def adiciona_ruido_sal_pimenta(img, quantidade=0.05):
    img_ruidosa = img.copy()
    h, w, c = img_ruidosa.shape
    num_ruido = int(quantidade * h * w)
    # Sal (255, 255, 255)
    coords_sal = [np.random.randint(0, i - 1, num_ruido) for i in (h, w)]
    img_ruidosa[coords_sal[0], coords_sal[1]] = [255, 255, 255]
    # Pimenta (0, 0, 0)
    coords_pimenta = [np.random.randint(0, i - 1, num_ruido) for i in (h, w)]
    img_ruidosa[coords_pimenta[0], coords_pimenta[1]] = [0, 0, 0]
    return img_ruidosa


def gera_imagem_ruidosa(tam_img=160, n_objetos=(1, 3), taxa_ruido=0.01, rng=None):
    rng = rng or random.Random()
    img_limpa = np.full((tam_img, tam_img, 3), 255, dtype=np.uint8)
    anotacoes = []
    for _ in range(rng.randint(*n_objetos)):
        classe_idx = rng.randrange(len(CLASSES))
        tamanho = rng.randint(tam_img // 10, tam_img // 5)
        cx = rng.randint(tamanho + 2, tam_img - tamanho - 2)
        cy = rng.randint(tamanho + 2, tam_img - tamanho - 2)
        rotacao = rng.uniform(0, 360)
        cor = tuple(rng.sample(range(30, 226), 3))
        x0, y0, x1, y1 = desenha_objeto(img_limpa, classe_idx, cx, cy, tamanho, rotacao, cor)
        x0, y0 = max(x0, 0), max(y0, 0)
        x1, y1 = min(x1, tam_img), min(y1, tam_img)
        xc, yc = (x0 + x1) / 2 / tam_img, (y0 + y1) / 2 / tam_img
        w, h = (x1 - x0) / tam_img, (y1 - y0) / tam_img
        anotacoes.append((classe_idx, xc, yc, w, h))
    img_ruidosa = adiciona_ruido_sal_pimenta(img_limpa, quantidade=taxa_ruido)
    return img_ruidosa, anotacoes

In [ ]:
# @fig-09-yolo-amostras-iniciais
# Geração de 5 amostras para exibição inicial do dataset sintético
n_amostras_iniciais = 5
rng_demo = random.Random(42)

imgs_demo = []
titulos_demo = []

for idx in range(n_amostras_iniciais):
    img_ruid, anotacoes = gera_imagem_ruidosa(rng=rng_demo)
    # Gravação temporária da anotação para leitura nativa por mm.showBoundBox
    filename_temp = f"temp_label_{idx}.txt"
    with open(filename_temp, "w") as f:
        for c, xc, yc, w, h in anotacoes:
            f.write(f"{c} {xc:.4f} {yc:.4f} {w:.4f} {h:.4f}\n")
    img_anotada = mm.showBoundBox(img_ruid, filename=filename_temp, fmt="yolo", show=False)
    imgs_demo.append(img_anotada)
    titulos_demo.append(f"Amostra {idx+1}")

# Exibição do painel de 5 amostras
mm.show(
    imgs_demo,
    titles=titulos_demo,
    cols=n_amostras_iniciais,
    figsize=(14, 3)
)

### Bloco 2: Montagem da Estrutura de Pastas e do Manifesto `data.yaml`

A biblioteca de treinamento empregada — a `ultralytics`, responsável pela implementação de referência do YOLOv8 (Jocher et al., 2023) — exige que as imagens e os arquivos de anotação estejam organizados segundo uma estrutura hierárquica específica, com diretórios distintos para os subconjuntos de treino (*train*) e de validação (*val*):

```text
shapes_dataset/
├── data.yaml
├── images/
│   ├── train/
│   └── val/
└── labels/
    ├── train/
    └── val/
```

O código a seguir grava 90 imagens ruidosas destinadas ao treino e 20 imagens destinadas à validação, cada uma acompanhada de um arquivo de texto com as anotações no formato YOLO descrito no bloco anterior. Em seguida, cria-se o manifesto `data.yaml`, arquivo de configuração que especifica os caminhos dos diretórios de treino e de validação e associa cada índice numérico ao nome da respectiva categoria — informação indispensável para que a biblioteca `ultralytics` interprete corretamente as anotações durante o treinamento.

In [ ]:
import os

base_dir = "shapes_dataset"
rng_global = random.Random(42)

for split, n_imgs in [("train", 90), ("val", 20)]:
    os.makedirs(f"{base_dir}/images/{split}", exist_ok=True)
    os.makedirs(f"{base_dir}/labels/{split}", exist_ok=True)
    for i in range(n_imgs):
        img_ruid, anotacoes = gera_imagem_ruidosa(rng=rng_global)
        cv2.imwrite(f"{base_dir}/images/{split}/{i:04d}.jpg", img_ruid)
        with open(f"{base_dir}/labels/{split}/{i:04d}.txt", "w") as f:
            for c, xc, yc, w, h in anotacoes:
                f.write(f"{c} {xc:.4f} {yc:.4f} {w:.4f} {h:.4f}\n")

with open(f"{base_dir}/data.yaml", "w") as f:
    f.write(f"path: {os.path.abspath(base_dir)}\ntrain: images/train\nval: images/val\nnames:\n")
    for i, nome in enumerate(CLASSES):
        f.write(f"  {i}: {nome}\n")

print("Dataset ruidoso gerado com sucesso: 90 imagens de treino e 20 de validação.")

### Bloco 3: Pré-processamento com Filtro de Mediana para Restauração da Imagem

Redes convolucionais profundas toleram parcialmente pequenas variações de textura, mas a presença de ruído impulsivo compromete a precisão da localização, pois pixels isolados de contraste máximo distorcem os gradientes de borda utilizados pelas camadas convolucionais iniciais. O **filtro de mediana** (`cv2.medianBlur`) constitui a solução clássica para essa categoria de degradação: cada pixel é substituído pela mediana dos valores contidos em uma vizinhança definida — neste caso, uma janela de $3 \times 3$ pixels —, o que elimina os valores extremos introduzidos pelo ruído sal e pimenta sem borrar as bordas dos objetos, ao contrário de filtros baseados em média (Gonzalez & Woods, 2018).

A @fig-09-yolo-pre-processamento apresenta o resultado da restauração aplicada sobre a primeira amostra do conjunto de treino, mantendo as anotações sobrepostas por meio da função `mm.showBoundBox()`.

In [ ]:
# @fig-09-yolo-pre-processamento
# 1. Carregamento da primeira amostra ruidosa do dataset
caminho_img = f"{base_dir}/images/train/0000.jpg"
caminho_label = f"{base_dir}/labels/train/0000.txt"

img_ruidosa = mm.read(caminho_img)

# 2. Pré-processamento com Filtro de Mediana (janela 3x3)
img_filtrada = cv2.medianBlur(img_ruidosa, ksize=3)

# 3. Sobreposição das bounding boxes com mm.showBoundBox
img_ruid_anotada = mm.showBoundBox(img_ruidosa, filename=caminho_label, fmt="yolo", show=False)
img_filt_anotada = mm.showBoundBox(img_filtrada, filename=caminho_label, fmt="yolo", show=False)

# 4. Exibição comparativa com mm.show
mm.show(
    [img_ruid_anotada, img_filt_anotada],
    titles=["1. Imagem Ruidosa Original (Sal e Pimenta)", "2. Pré-processada (Filtro de Mediana 3x3)"],
    cols=2,
    figsize=(9, 4)
)

### Bloco 4: Restauração do *Dataset* e Treinamento do YOLOv8

Estabelecida a eficácia do filtro de mediana sobre uma única amostra, a mesma operação é aplicada em lote sobre a totalidade das imagens dos diretórios `train` e `val`, precedendo o ajuste fino do detector. Em seguida, instancia-se o modelo `yolov8n.pt` — variante compacta da arquitetura YOLOv8, pré-treinada no conjunto COCO — e executa-se o ajuste fino por meio do método `.train()`, configurando a resolução de entrada para $320 \times 320$ pixels ao longo de 30 épocas (*epochs*), sem congelamento do *backbone* (o conjunto de camadas convolucionais responsáveis pela extração de características), de modo que os filtros originalmente aprendidos em fotografias naturais possam se adaptar às geometrias sintéticas do novo domínio.

O desempenho do modelo é avaliado, ao final do treinamento, pelo método `.val()`, que calcula três métricas sobre o conjunto de validação restaurado: a precisão (*precision*), definida pela razão entre detecções corretas e o total de detecções realizadas; a revocação (*recall*), definida pela razão entre detecções corretas e o total de objetos presentes na imagem; e o $mAP@50$ (*mean Average Precision*, com limiar de sobreposição de 50% entre a caixa predita e a caixa verdadeira), métrica consolidada como referência para a avaliação comparativa de detectores de objetos (Everingham et al., 2010).

In [ ]:
# eval: false
import os
import cv2
import torch
from ultralytics import YOLO

base_dir = "shapes_dataset"
device = "cuda" if torch.cuda.is_available() else "cpu"

# 1. Aplicação do Filtro de Mediana em lote sobre as pastas do dataset
for split in ["train", "val"]:
    pasta_imgs = f"{base_dir}/images/{split}"
    for nome_arq in os.listdir(pasta_imgs):
        if nome_arq.endswith(".jpg"):
            caminho_completo = os.path.join(pasta_imgs, nome_arq)
            img_ruidosa = cv2.imread(caminho_completo)
            img_filtrada = cv2.medianBlur(img_ruidosa, ksize=3)
            cv2.imwrite(caminho_completo, img_filtrada)

print("Pré-processamento em lote (Filtro de Mediana) concluído em train e val.\n")

# 2. Carregamento do modelo YOLOv8n pré-treinado na COCO
modelo_yolo = YOLO("yolov8n.pt")

# 3. Treinamento (Fine-Tuning) no dataset geométrico restaurado
resultados_treino = modelo_yolo.train(
    data=f"{base_dir}/data.yaml",
    epochs=30,
    imgsz=320,
    batch=16,
    device=device,
    verbose=False,
    plots=False
)

# 4. Avaliação quantitativa no conjunto de validação
metricas = modelo_yolo.val()

precision = metricas.results_dict["metrics/precision(B)"]
recall = metricas.results_dict["metrics/recall(B)"]
map50 = metricas.results_dict["metrics/mAP50(B)"]

print("--- Desempenho Otimizado do Modelo YOLOv8 ---")
print(f"Precisão: {precision*100:.2f}%")
print(f"Revocação (Recall): {recall*100:.2f}%")
print(f"mAP@50 (Acurácia de Detecção): {map50*100:.2f}%")

### Bloco 5: Comparativo de Inferência entre Imagem Ruidosa e Imagem Restaurada

Para evidenciar o impacto do ruído sal e pimenta sobre a capacidade de localização do detector treinado, realiza-se a inferência comparativa sobre uma imagem do conjunto de validação em duas condições: sem qualquer tratamento e após a aplicação pontual do filtro de mediana. O método `.predict()` executa a detecção sobre ambas as versões da imagem, com limiar de confiança fixado em 25% (`conf=0.25`), e a @fig-09-yolo-inferencia-comparativa exibe as caixas delimitadoras e os rótulos preditos em cada uma das condições.

In [ ]:
# eval: false
# @fig-09-yolo-inferencia-comparativa
# 1. Carregamento de uma amostra de teste original (sem o filtro salvo em lote)
caminho_teste = f"{base_dir}/images/val/0001.jpg"
img_ruidosa_teste = mm.read(caminho_teste)

# 2. Aplicação pontual do Filtro de Mediana (3x3) para comparação
img_filtrada_teste = cv2.medianBlur(img_ruidosa_teste, ksize=3)

# 3. Inferência com o modelo YOLOv8 treinado
pred_ruidosa = modelo_yolo.predict(img_ruidosa_teste, conf=0.25, verbose=False)[0]
pred_filtrada = modelo_yolo.predict(img_filtrada_teste, conf=0.25, verbose=False)[0]

# 4. Extração das matrizes anotadas pelo gerador do YOLO (conversão BGR -> RGB)
img_pred_ruid = cv2.cvtColor(pred_ruidosa.plot(), cv2.COLOR_BGR2RGB)
img_pred_filt = cv2.cvtColor(pred_filtrada.plot(), cv2.COLOR_BGR2RGB)

# 5. Exibição comparativa padronizada via mm.show
mm.show(
    [img_pred_ruid, img_pred_filt],
    titles=[
        f"Inferência na Imagem Ruidosa ({len(pred_ruidosa.boxes)} objetos)",
        f"Inferência na Imagem Filtrada ({len(pred_filtrada.boxes)} objetos)"
    ],
    cols=2,
    figsize=(10, 4)
)

:::{.callout-note}

#### Um domínio muito mais distante que o dos dígitos {.unnumbered}

No experimento de transferência de aprendizado entre dígitos manuscritos (domínios A e B), a tarefa de origem e a de destino compartilhavam estatísticas visuais próximas: ambas consistiam em traços em tons de cinza sobre fundo uniforme. Aqui, a distância entre domínios é consideravelmente maior — o YOLO foi pré-treinado em fotografias naturais coloridas da COCO (pessoas, animais, veículos, objetos do cotidiano), e a tarefa de destino consiste em formas geométricas sintéticas, de cor sólida e contorno bem definido, sem textura, iluminação ou fundo complexo.

Ainda assim, a transferência de aprendizado permanece vantajosa: as camadas iniciais de um detector treinado na COCO aprendem filtros genéricos — detectores de borda, de canto e de regiões de contraste — que continuam úteis para delimitar o contorno de um triângulo ou de uma estrela, mesmo quando o conteúdo visual final é bastante distinto (Yosinski et al., 2014). É por essa razão que o ajuste fino de todas as camadas, combinado ao pré-processamento para eliminação do ruído sal e pimenta, viabiliza taxas de acurácia elevadas na detecção de objetos do novo domínio.
:::

### Referências

- EVERINGHAM, M.; VAN GOOL, L.; WILLIAMS, C. K. I.; WINN, J.; ZISSERMAN, A. **The Pascal Visual Object Classes (VOC) Challenge**. International Journal of Computer Vision, v. 88, n. 2, p. 303–338, 2010.
- GONZALEZ, R. C.; WOODS, R. E. **Digital Image Processing**. 4. ed. New York: Pearson, 2018.
- JOCHER, G.; CHAURASIA, A.; QIU, J. **Ultralytics YOLOv8**. 2023. Disponível em: https://github.com/ultralytics/ultralytics.
- LIN, T.-Y.; MAIRE, M.; BELONGIE, S.; HAYS, J.; PERONA, P.; RAMANAN, D.; DOLLÁR, P.; ZITNICK, C. L. **Microsoft COCO: Common Objects in Context**. In: European Conference on Computer Vision (ECCV), 2014.
- PAN, S. J.; YANG, Q. **A Survey on Transfer Learning**. IEEE Transactions on Knowledge and Data Engineering, v. 22, n. 10, p. 1345–1359, 2010.
- REDMON, J.; DIVVALA, S.; GIRSHICK, R.; FARHADI, A. **You Only Look Once: Unified, Real-Time Object Detection**. In: IEEE Conference on Computer Vision and Pattern Recognition (CVPR), 2016.
- REN, S.; HE, K.; GIRSHICK, R.; SUN, J. **Faster R-CNN: Towards Real-Time Object Detection with Region Proposal Networks**. In: Advances in Neural Information Processing Systems (NeurIPS), 2015.
- YOSINSKI, J.; CLUNE, J.; BENGIO, Y.; LIPSON, H. **How Transferable Are Features in Deep Neural Networks?**. In: Advances in Neural Information Processing Systems (NeurIPS), 2014.